In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/BTT_Amazon1A_Studio')

In [ ]:
print(os.listdir('/content/drive/MyDrive/BTT_Amazon1A_Studio'))

In [ ]:
excel_data = pd.read_excel('online_retail_II.xlsx', sheet_name=None)
print("Found sheets:", excel_data.keys())

# Combine both sheets into a single DataFrame
df = pd.concat(excel_data.values(), ignore_index=True)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
null_counts = df.isnull().sum().sort_values(ascending=False)
print(null_counts)

In [ ]:
df = df.dropna(subset=['Customer ID'])
df["Customer ID"] = df['Customer ID']

In [ ]:
#drop rows with null for customer ID
nullcount = df['Customer ID'].isnull().sum()
print(nullcount)

In [ ]:
df['Description'] = df['Description'].fillna('Unknown')

In [ ]:
nullcount = df['Description'].isnull().sum()
print(nullcount)

In [ ]:
#drop cancelled invoices
#df = df[~df['Invoice'].astype(str).str.startswith('C')]

#need to confirm with coach and CA

In [ ]:
df['Invoice'].astype(str).str.startswith('C').sum()

In [ ]:
#convert timestamps
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour
df['Minute'] = df['InvoiceDate'].dt.minute
df = df.drop(['InvoiceDate'], axis=1)

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
#convert stock code and customer ID to string
df['Customer ID'] = df['Customer ID'].astype(float).astype(int).astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df = df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
#df_combined = (
#    df.groupby(['Customer ID', 'Invoice'], as_index=False)
#    .agg(
#        {
#            'Quantity': 'sum',
#            #'Total_Amount': 'sum',  # Include any other numeric columns to sum
#        }
#    )
#)

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df[['Year', 'Month', 'Day']])

In [ ]:
df['Revenue'] = df['Quantity'] * df['Price']

In [ ]:
df['SnapshotMonth'] = df['InvoiceDate'].dt.to_period('M') #add dt.to_timestamp for end date of month possibly

In [ ]:
monthly_data = df.groupby(['Customer ID', 'SnapshotMonth']).agg(
    month_orders=('Invoice', 'nunique'),
    month_revenue=('Revenue', 'sum'),
    month_quantity=('Quantity', 'sum'),
    month_diff_items=('StockCode', 'nunique'),
    month_last_order=('InvoiceDate', 'max')

).reset_index()#should add cancellation column in here once figure out what to do with that?

In [ ]:
min_period = monthly_data['SnapshotMonth'].min()
max_period = monthly_data['SnapshotMonth'].max()
print(min_period)
print(max_period)

In [ ]:
all_periods = pd.period_range(start=min_period, end=max_period, freq='M')

In [ ]:
customer_first_period = df.groupby('Customer ID')['SnapshotMonth'].min()

records=[]
for customer_id, first_period in customer_first_period.items():
    valid_periods = all_periods[all_periods >= first_period]
    for period in valid_periods:
        records.append((customer_id, period))
feature_table = pd.DataFrame(records, columns=['Customer ID', 'SnapshotMonth'])

In [ ]:
feature_table = feature_table.merge(monthly_data, on=['Customer ID', 'SnapshotMonth'], how='left')

In [ ]:
feature_table.head(190)

In [ ]:
purchase_dates = df.groupby('Customer ID')['InvoiceDate'].unique()

In [ ]:
data_end = df['InvoiceDate'].max()

churn = []
for customer_id, period in zip(feature_table['Customer ID'], feature_table['SnapshotMonth']):
    month_end = period.end_time
    window_end = month_end + pd.Timedelta(days=90)
    if window_end > data_end:
        churn.append(None)
        continue
    dates = purchase_dates[customer_id]
    bought = ((dates > month_end) & (dates <= window_end)).any()
    churn.append(0 if bought else 1)

feature_table['churn'] = churn

In [ ]:
labeled_table = feature_table.dropna(subset=['churn']).copy()
labeled_table['churn'] = labeled_table['churn'].astype(int)

In [ ]:
# explore the data
labeled_table.describe(include=all)

In [ ]:
# plot features using a box plot
for col in ['month_orders', 'month_revenue', 'month_quantity', 'month_diff_items']:
    sns.boxplot(data=labeled_table, x='churn', y=col, showfliers=False)
    plt.title(col)
    plt.show()

In [ ]:
labeled_table.groupby('churn').mean(numeric_only=True)